In [ ]:
from google.colab import files
import pandas as pd

**Extract The Data**

In [4]:
cust_upload = files.upload('MVNO Data/')

Saving customer.json to MVNO Data/customer.json


In [5]:
plan_upload = files.upload('MVNO Data/')

Saving plan.json to MVNO Data/plan.json


In [300]:
billing_upload = files.upload('MVNO Data/')

Saving updated_billing_1000.json to MVNO Data/updated_billing_1000.json


In [301]:
usage_upload = files.upload('MVNO Data/')

Saving updated_usage_1000.json to MVNO Data/updated_usage_1000.json


**Converts the uploaded data into DataFrames**

In [471]:
customer = pd.read_json('MVNO Data/customer.json')
plan = pd.read_json('MVNO Data/plan.json')
billing = pd.read_json('MVNO Data/billing.json')
usage = pd.read_json('MVNO Data/usage.json')

In [472]:
customer

,Customer_ID,Customer_Name,Customer_Segment,Join_Date
0,C001,Arun Kumar,Prepaid,2024-01-15
1,C002,Priya Sharma,Postpaid,2023-11-20
2,C003,Rahul Das,Prepaid,2025-06-10
3,C004,Sneha Iyer,Postpaid,2022-09-01
4,C005,Karthik R,Prepaid,2024-07-19
5,C006,Meena S,Postpaid,2023-02-25


In [473]:
plan

,Plan_ID,Plan_Name,Plan_Type,Monthly_Price,Data_Limit_GB,Voice_Limit_Minutes
0,P101,Combo Saver,Combo,299,1.5,500.0
1,P102,Data Max,Data,499,3.0,NaN
2,P103,Voice Basic,Voice,199,0.5,800.0
3,P104,Unlimited Pro,Combo,799,5.0,2000.0


In [474]:
plan['Voice_Limit_Minutes'] = plan['Voice_Limit_Minutes'].fillna(1500.0)

plan

,Plan_ID,Plan_Name,Plan_Type,Monthly_Price,Data_Limit_GB,Voice_Limit_Minutes
0,P101,Combo Saver,Combo,299,1.5,500.0
1,P102,Data Max,Data,499,3.0,1500.0
2,P103,Voice Basic,Voice,199,0.5,800.0
3,P104,Unlimited Pro,Combo,799,5.0,2000.0


In [475]:
billing

,Billing_ID,Customer_ID,Plan_ID,Bill_Date,Total_Amount,Discount_Percentage,Final_Amount,Payment_Status,Payment_Method
0,B001,C001,P101,2026-03-31,299,10.0,269.10,Paid,UPI
1,B002,C002,P102,2026-03-31,550,5.0,522.50,Paid,Card
2,B003,C003,P103,2026-03-31,250,NaN,250.00,Pending,UPI
3,B004,C004,P104,2026-03-31,799,15.0,679.15,Failed,NetBanking
4,B005,C005,P101,2026-03-31,299,0.0,299.00,Paid,None
5,B006,C006,P102,2026-03-31,499,8.0,459.08,Paid,Card


In [476]:
billing['Discount_Percentage']=billing['Discount_Percentage'].fillna(5.0)

billing['Payment_Method'] = billing['Payment_Method'].fillna('Unknown')

billing.isnull().sum()

,0
Billing_ID,0
Customer_ID,0
Plan_ID,0
Bill_Date,0
Total_Amount,0
Discount_Percentage,0
Final_Amount,0
Payment_Status,0
Payment_Method,0


In [477]:
usage

,Usage_ID,Customer_ID,Usage_Date,Data_Used_GB,Voice_Used_Minutes
0,U001,C001,2026-03-01,1.2,450.0
1,U002,C002,2026-03-05,3.5,120.0
2,U003,C003,2026-03-10,NaN,900.0
3,U004,C004,2026-03-12,4.2,1800.0
4,U005,C005,2026-03-15,0.8,300.0
5,U006,C006,2026-03-20,2.9,NaN


In [478]:
usage['Data_Used_GB'] = usage['Data_Used_GB'].fillna(usage['Data_Used_GB'].mean())
usage['Voice_Used_Minutes'] = usage['Voice_Used_Minutes'].fillna(usage['Voice_Used_Minutes'].mean())

In [479]:
usage

,Usage_ID,Customer_ID,Usage_Date,Data_Used_GB,Voice_Used_Minutes
0,U001,C001,2026-03-01,1.20,450.0
1,U002,C002,2026-03-05,3.50,120.0
2,U003,C003,2026-03-10,2.52,900.0
3,U004,C004,2026-03-12,4.20,1800.0
4,U005,C005,2026-03-15,0.80,300.0
5,U006,C006,2026-03-20,2.90,714.0


In [480]:
usage.isnull().sum()

,0
Usage_ID,0
Customer_ID,0
Usage_Date,0
Data_Used_GB,0
Voice_Used_Minutes,0


## 💰 Revenue & Billing Analysis

In [481]:
# 1.	What is the total revenue generated over time (daily/monthly)?
billing['Bill_Date'] = pd.to_datetime(billing['Bill_Date'])
billing['day'] = billing['Bill_Date'].dt.day

In [482]:
#Change the bill date for shows the variations of data
billing.loc[0,'Bill_Date']='2026-02-01 00:00:00'

In [483]:
billing['month'] = billing['Bill_Date'].dt.month

In [484]:
day_wise_overall_revenue = billing.groupby('day').agg(
    Total_Revenue = ('Final_Amount','sum')
)

In [485]:
day_wise_overall_revenue

,Total_Revenue
day,
31,2478.83


In [486]:
month_wise_overall_revenue = billing.groupby('month').agg(
    Total_Revenue = ('Final_Amount','sum')
)

In [487]:
month_wise_overall_revenue

,Total_Revenue
month,
2,269.10
3,2209.73


In [488]:
merge_customer_billing = pd.merge(billing,customer,on='Customer_ID',how='left')

In [489]:
merge_customer_billing

,Billing_ID,Customer_ID,Plan_ID,Bill_Date,Total_Amount,Discount_Percentage,Final_Amount,Payment_Status,Payment_Method,day,month,Customer_Name,Customer_Segment,Join_Date
0,B001,C001,P101,2026-02-01,299,10.0,269.10,Paid,UPI,31,2,Arun Kumar,Prepaid,2024-01-15
1,B002,C002,P102,2026-03-31,550,5.0,522.50,Paid,Card,31,3,Priya Sharma,Postpaid,2023-11-20
2,B003,C003,P103,2026-03-31,250,5.0,250.00,Pending,UPI,31,3,Rahul Das,Prepaid,2025-06-10
3,B004,C004,P104,2026-03-31,799,15.0,679.15,Failed,NetBanking,31,3,Sneha Iyer,Postpaid,2022-09-01
4,B005,C005,P101,2026-03-31,299,0.0,299.00,Paid,Unknown,31,3,Karthik R,Prepaid,2024-07-19
5,B006,C006,P102,2026-03-31,499,8.0,459.08,Paid,Card,31,3,Meena S,Postpaid,2023-02-25


In [490]:
# 2.	Which customers generate the highest revenue? (Top 3 customers)
cust_revenue = merge_customer_billing.groupby('Customer_Name').agg(
    Total_Revenue = ('Final_Amount','sum')
)

In [491]:
cust_revenue.sort_values('Total_Revenue',ascending=False).head(3)

,Total_Revenue
Customer_Name,
Sneha Iyer,679.15
Priya Sharma,522.50
Meena S,459.08


In [492]:
# 3.	What is the average revenue per customer (ARPU)?
# Formula Total Revenue generated / Number of customers
total_revenue = merge_customer_billing['Final_Amount'].sum()

number_of_customers = merge_customer_billing['Customer_ID'].nunique()

ARPU = total_revenue/number_of_customers

print("ARPU :",ARPU)

ARPU : 413.1383333333333


In [493]:
# 4.	How does discount impact final revenue?
discount_impact = merge_customer_billing.groupby('Total_Amount').agg(
    Avg_Disc_Percentage = ('Discount_Percentage','mean'),
    Avg_Revenue = ('Final_Amount','mean')
)

In [494]:
discount_impact[['Avg_Disc_Percentage','Avg_Revenue']].corr()

# Here we can see strong positive (0.79906) correlation b/w Avg discount % and Avg Revenue. Therefore the discount increases the revenue as per our data

,Avg_Disc_Percentage,Avg_Revenue
Avg_Disc_Percentage,1.000000,0.803832
Avg_Revenue,0.803832,1.000000


In [495]:
# 5.	Which payment method contributes the most revenue?
Payment_Method_Wise_Revenue = merge_customer_billing.groupby('Payment_Method').agg(
    Revenue = ('Final_Amount','sum')
)

Payment_Method_Wise_Revenue.sort_values('Revenue',ascending=False)

# Highest Revenue from Card (Rs.981.58)

,Revenue
Payment_Method,
Card,981.58
NetBanking,679.15
UPI,519.10
Unknown,299.00


In [496]:
# 6.	What % of bills are paid vs pending vs failed?

total_transaction = merge_customer_billing['Billing_ID'].count()

Payment_Status_Analysis = merge_customer_billing.groupby('Payment_Status').agg(
    Count = ('Billing_ID','count')
)

Payment_Status_Analysis['Percentage %'] = Payment_Status_Analysis['Count']/total_transaction


In [497]:
print("Total Transactions(billing) :",total_transaction)

Total Transactions(billing) : 6


In [498]:
Payment_Status_Analysis.sort_values('Percentage %',ascending=False)

,Count,Percentage %
Payment_Status,,
Paid,4,0.666667
Failed,1,0.166667
Pending,1,0.166667


## 📱 Plan Performance Analysis

In [499]:
# 7.	Which plan generates the highest revenue?
merge_customer_billing_plan = pd.merge(merge_customer_billing,plan,on='Plan_ID',how='left')
merged_data = merge_customer_billing_plan
merged_data

,Billing_ID,Customer_ID,Plan_ID,Bill_Date,Total_Amount,Discount_Percentage,Final_Amount,Payment_Status,Payment_Method,day,month,Customer_Name,Customer_Segment,Join_Date,Plan_Name,Plan_Type,Monthly_Price,Data_Limit_GB,Voice_Limit_Minutes
0,B001,C001,P101,2026-02-01,299,10.0,269.10,Paid,UPI,31,2,Arun Kumar,Prepaid,2024-01-15,Combo Saver,Combo,299,1.5,500.0
1,B002,C002,P102,2026-03-31,550,5.0,522.50,Paid,Card,31,3,Priya Sharma,Postpaid,2023-11-20,Data Max,Data,499,3.0,1500.0
2,B003,C003,P103,2026-03-31,250,5.0,250.00,Pending,UPI,31,3,Rahul Das,Prepaid,2025-06-10,Voice Basic,Voice,199,0.5,800.0
3,B004,C004,P104,2026-03-31,799,15.0,679.15,Failed,NetBanking,31,3,Sneha Iyer,Postpaid,2022-09-01,Unlimited Pro,Combo,799,5.0,2000.0
4,B005,C005,P101,2026-03-31,299,0.0,299.00,Paid,Unknown,31,3,Karthik R,Prepaid,2024-07-19,Combo Saver,Combo,299,1.5,500.0
5,B006,C006,P102,2026-03-31,499,8.0,459.08,Paid,Card,31,3,Meena S,Postpaid,2023-02-25,Data Max,Data,499,3.0,1500.0


In [500]:
Plan_Analysis = merged_data.groupby('Plan_Name').agg(
    Revenue = ('Final_Amount','sum')
)

In [501]:
Plan_Analysis.sort_values('Revenue',ascending=False).head(1)

,Revenue
Plan_Name,
Data Max,981.58


In [502]:
# 8.	Which plan has the most subscribers?
Plan_wise_subs_count = merged_data.groupby('Plan_Name').agg(
    Subscriber_Count = ('Customer_ID','count')
)

Plan_wise_subs_count

,Subscriber_Count
Plan_Name,
Combo Saver,2
Data Max,2
Unlimited Pro,1
Voice Basic,1


In [503]:
Final_Data = pd.merge(merged_data,usage,on='Customer_ID',how='left')

In [504]:
# 9.	Are customers fully utilizing their data/voice limits?
Final_Data[['Customer_Name','Voice_Limit_Minutes','Voice_Used_Minutes','Data_Limit_GB','Data_Used_GB']]

,Customer_Name,Voice_Limit_Minutes,Voice_Used_Minutes,Data_Limit_GB,Data_Used_GB
0,Arun Kumar,500.0,450.0,1.5,1.20
1,Priya Sharma,1500.0,120.0,3.0,3.50
2,Rahul Das,800.0,900.0,0.5,2.52
3,Sneha Iyer,2000.0,1800.0,5.0,4.20
4,Karthik R,500.0,300.0,1.5,0.80
5,Meena S,1500.0,714.0,3.0,2.90


In [505]:
Usage_Calc = Final_Data[['Customer_Name','Voice_Limit_Minutes','Voice_Used_Minutes','Data_Limit_GB','Data_Used_GB']]

In [506]:
Usage_Calc['Voice Usage Utilization'] = Usage_Calc['Voice_Limit_Minutes']-Usage_Calc['Voice_Used_Minutes']
Usage_Calc['Data Usage Utilization'] = Usage_Calc['Data_Limit_GB']-Usage_Calc['Data_Used_GB']

/tmp/ipykernel_470/1180537731.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Usage_Calc['Voice Usage Utilization'] = Usage_Calc['Voice_Limit_Minutes']-Usage_Calc['Voice_Used_Minutes']
/tmp/ipykernel_470/1180537731.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Usage_Calc['Data Usage Utilization'] = Usage_Calc['Data_Limit_GB']-Usage_Calc['Data_Used_GB']


In [507]:
Usage_Calc

,Customer_Name,Voice_Limit_Minutes,Voice_Used_Minutes,Data_Limit_GB,Data_Used_GB,Voice Usage Utilization,Data Usage Utilization
0,Arun Kumar,500.0,450.0,1.5,1.20,50.0,0.30
1,Priya Sharma,1500.0,120.0,3.0,3.50,1380.0,-0.50
2,Rahul Das,800.0,900.0,0.5,2.52,-100.0,-2.02
3,Sneha Iyer,2000.0,1800.0,5.0,4.20,200.0,0.80
4,Karthik R,500.0,300.0,1.5,0.80,200.0,0.70
5,Meena S,1500.0,714.0,3.0,2.90,786.0,0.10


In [508]:
def VoiceUtilize(x):
  if x == 0:
    return "Utilized Completely"
  elif x < 0:
    return "Utilized Completely + Add On"
  else:
    return "Partially Utilized"

Usage_Calc['Is Voice Limit reached?'] = Usage_Calc['Voice Usage Utilization'].apply(VoiceUtilize)

/tmp/ipykernel_470/3959472434.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Usage_Calc['Is Voice Limit reached?'] = Usage_Calc['Voice Usage Utilization'].apply(VoiceUtilize)


In [509]:
Voice_Usage = Usage_Calc[['Customer_Name','Voice_Limit_Minutes','Voice_Used_Minutes','Is Voice Limit reached?']]

Voice_Usage

,Customer_Name,Voice_Limit_Minutes,Voice_Used_Minutes,Is Voice Limit reached?
0,Arun Kumar,500.0,450.0,Partially Utilized
1,Priya Sharma,1500.0,120.0,Partially Utilized
2,Rahul Das,800.0,900.0,Utilized Completely + Add On
3,Sneha Iyer,2000.0,1800.0,Partially Utilized
4,Karthik R,500.0,300.0,Partially Utilized
5,Meena S,1500.0,714.0,Partially Utilized


In [510]:
def DataUtilize(x):
  if x == 0:
    return "Utilized Completely"
  elif x < 0:
    return "Overusing"
  else:
    return "Partially Utilized"

Usage_Calc['Is Data Limit reached?'] = Usage_Calc['Data Usage Utilization'].apply(DataUtilize)

/tmp/ipykernel_470/2879739681.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Usage_Calc['Is Data Limit reached?'] = Usage_Calc['Data Usage Utilization'].apply(DataUtilize)


In [511]:
Data_Usage = Usage_Calc[['Customer_Name','Data_Limit_GB','Data_Used_GB','Is Data Limit reached?']]

Data_Usage

,Customer_Name,Data_Limit_GB,Data_Used_GB,Is Data Limit reached?
0,Arun Kumar,1.5,1.20,Partially Utilized
1,Priya Sharma,3.0,3.50,Overusing
2,Rahul Das,0.5,2.52,Overusing
3,Sneha Iyer,5.0,4.20,Partially Utilized
4,Karthik R,1.5,0.80,Partially Utilized
5,Meena S,3.0,2.90,Partially Utilized


In [512]:
# 10.	Which plan has high revenue but low usage (overpricing opportunity)?
Plan_wise_usage = Final_Data.groupby('Plan_Name').agg(
    Revenue = ('Final_Amount','sum'),
    Voice_Usage = ('Voice_Used_Minutes','sum'),
    Data_Usage = ('Data_Used_GB','sum')
)

Plan_wise_usage['Total_Usage'] = (Plan_wise_usage['Voice_Usage']+Plan_wise_usage['Data_Usage'])

Plan_wise_usage['Revenue_Per_Usage'] = Plan_wise_usage['Revenue']/Plan_wise_usage['Total_Usage']

Plan_wise_usage.sort_values('Revenue_Per_Usage',ascending=False)

# Data Max is the plan which has high revenue but low usage

,Revenue,Voice_Usage,Data_Usage,Total_Usage,Revenue_Per_Usage
Plan_Name,,,,,
Data Max,981.58,834.0,6.40,840.40,1.167991
Combo Saver,568.10,750.0,2.00,752.00,0.755452
Unlimited Pro,679.15,1800.0,4.20,1804.20,0.376427
Voice Basic,250.00,900.0,2.52,902.52,0.277002


## **📶 Usage Behavior Analysis**

In [513]:
# 11.	What is the average data usage per customer?
Avg_Data_Usage_By_Customer = Final_Data.groupby('Customer_Name').agg(
    Avg_Data_Usage = ('Data_Used_GB','mean')
)

Avg_Data_Usage_By_Customer.sort_values('Avg_Data_Usage',ascending=False)

,Avg_Data_Usage
Customer_Name,
Sneha Iyer,4.20
Priya Sharma,3.50
Meena S,2.90
Rahul Das,2.52
Arun Kumar,1.20
Karthik R,0.80


In [514]:
# 12.	Which customers have high usage but low billing (potential loss)?

In [515]:
Customer_Usage = Final_Data.groupby('Customer_Name').agg(
    Data_Usage = ('Data_Used_GB','sum'),
    Voice_Usage = ('Voice_Used_Minutes','sum'),
    billing_amount = ('Final_Amount','sum')
)

Customer_Usage['Overall_Usage'] = Customer_Usage['Data_Usage']+ Customer_Usage['Voice_Usage']

Customer_Usage['Score'] = Customer_Usage['Overall_Usage']/Customer_Usage['billing_amount']

Customer_Usage.sort_values('Score',ascending=False)

# Rahul Das is the person have high usage but low billing

,Data_Usage,Voice_Usage,billing_amount,Overall_Usage,Score
Customer_Name,,,,,
Rahul Das,2.52,900.0,250.00,902.52,3.610080
Sneha Iyer,4.20,1800.0,679.15,1804.20,2.656556
Arun Kumar,1.20,450.0,269.10,451.20,1.676700
Meena S,2.90,714.0,459.08,716.90,1.561601
Karthik R,0.80,300.0,299.00,300.80,1.006020
Priya Sharma,3.50,120.0,522.50,123.50,0.236364


In [516]:
# 13.	What % of users(Voice) are heavy vs moderate vs low users?
#def UserCategory(x):

user_category = Final_Data[['Customer_Name','Voice_Used_Minutes']]

quant75 = user_category['Voice_Used_Minutes'].quantile(0.75)
quant40 = user_category['Voice_Used_Minutes'].quantile(0.40)

def UserCategory(x):
  if x >= quant75:
    return 'Heavy User'
  elif x >= quant40:
    return 'Moderate User'
  else:
    return 'Low User'

user_category['Category(Voice Usage)'] = user_category['Voice_Used_Minutes'].apply(UserCategory)

user_category


/tmp/ipykernel_470/3599081078.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_category['Category(Voice Usage)'] = user_category['Voice_Used_Minutes'].apply(UserCategory)


,Customer_Name,Voice_Used_Minutes,Category(Voice Usage)
0,Arun Kumar,450.0,Moderate User
1,Priya Sharma,120.0,Low User
2,Rahul Das,900.0,Heavy User
3,Sneha Iyer,1800.0,Heavy User
4,Karthik R,300.0,Low User
5,Meena S,714.0,Moderate User


In [517]:
# 14.	How does usage vary across plan types (Data vs Combo Vs Voice)?
usage_pivot = Final_Data.pivot_table(index='Plan_Type',columns='Plan_Name',values='Voice_Used_Minutes',aggfunc='sum').fillna(0)

usage_pivot


Plan_Name,Combo Saver,Data Max,Unlimited Pro,Voice Basic
Plan_Type,,,,
Combo,750.0,0.0,1800.0,0.0
Data,0.0,834.0,0.0,0.0
Voice,0.0,0.0,0.0,900.0


In [518]:
# 15.	Which customer segment (Prepaid/Postpaid) generates more revenue?
customer_segment_wise_analysis = Final_Data.groupby('Customer_Segment').agg(
    Revenue = ('Final_Amount','sum')
)

customer_segment_wise_analysis.sort_values('Revenue',ascending=False)


# Postpaid generates more revenue (Rs.1660.73)

,Revenue
Customer_Segment,
Postpaid,1660.73
Prepaid,818.10


In [519]:
# 16.	What is the average monthly spend by customer segment?
avg_monthly_spend_by_cust = Final_Data.pivot_table(index='Customer_Name',columns='month',values='Final_Amount',aggfunc='sum').fillna(0)

avg_monthly_spend_by_cust

month,2,3
Customer_Name,,
Arun Kumar,269.1,0.00
Karthik R,0.0,299.00
Meena S,0.0,459.08
Priya Sharma,0.0,522.50
Rahul Das,0.0,250.00
Sneha Iyer,0.0,679.15


In [520]:
# 17.	Identify inactive or low-usage customers
user_category[user_category['Category(Voice Usage)']=='Low User']

,Customer_Name,Voice_Used_Minutes,Category(Voice Usage)
1,Priya Sharma,120.0,Low User
4,Karthik R,300.0,Low User


In [521]:
# 18.	What is the relationship between usage and billing amount? (correlation 🔥)
Customer_Usage[['Overall_Usage','billing_amount']].corr()

# positive correlation (0.545409) b/w Overall Usage & Billing Amount. While Increaing the Usage the billing amount as well increasing

,Overall_Usage,billing_amount
Overall_Usage,1.000000,0.545409
billing_amount,0.545409,1.000000


In [522]:
# 19.	Identify customers who are overusing plan limits → upsell opportunity
Usage_Calc.loc[(Usage_Calc['Voice Usage Utilization']<0) | (Usage_Calc['Data Usage Utilization']<0)]

# Priya Sharma & Rahul Das overusing plan limits

,Customer_Name,Voice_Limit_Minutes,Voice_Used_Minutes,Data_Limit_GB,Data_Used_GB,Voice Usage Utilization,Data Usage Utilization,Is Voice Limit reached?,Is Data Limit reached?
1,Priya Sharma,1500.0,120.0,3.0,3.50,1380.0,-0.50,Partially Utilized,Overusing
2,Rahul Das,800.0,900.0,0.5,2.52,-100.0,-2.02,Utilized Completely + Add On,Overusing


In [523]:
# 20. Detect customers who are underusing → downgrade recommendation
Usage_Calc[(Usage_Calc['Is Data Limit reached?']=='Partially Utilized') & (Usage_Calc['Is Voice Limit reached?']=='Partially Utilized')].sort_values(['Voice Usage Utilization','Data Usage Utilization'],ascending=True)

,Customer_Name,Voice_Limit_Minutes,Voice_Used_Minutes,Data_Limit_GB,Data_Used_GB,Voice Usage Utilization,Data Usage Utilization,Is Voice Limit reached?,Is Data Limit reached?
0,Arun Kumar,500.0,450.0,1.5,1.2,50.0,0.3,Partially Utilized,Partially Utilized
4,Karthik R,500.0,300.0,1.5,0.8,200.0,0.7,Partially Utilized,Partially Utilized
3,Sneha Iyer,2000.0,1800.0,5.0,4.2,200.0,0.8,Partially Utilized,Partially Utilized
5,Meena S,1500.0,714.0,3.0,2.9,786.0,0.1,Partially Utilized,Partially Utilized
